# Training SFPT on Tiny Shakespeare

This notebook demonstrates how to train the **Sparse Fourier Phase Transformer (SFPT)** on the **Tiny Shakespeare** dataset for character-level text generation.

In [ ]:
import os
import sys

# Add parent directory to path
current_dir = os.getcwd()

print(f"Current Working Directory: {current_dir}")
print("Files in current directory:")
print(os.listdir(current_dir))

def setup_environment():
    # Check if running in Colab
    try:
        import google.colab
        IN_COLAB = True
    except ImportError:
        IN_COLAB = False

    if IN_COLAB:
        print("\n[INFO] Detected Google Colab environment.")
        # Check if nano_moe is present
        if not os.path.exists('nano_moe'):
            print("[WARNING] 'nano_moe' package NOT found in current directory.")
            print("You are likely connected to a remote runtime that does not have your local files.")
            print("Please upload 'nano_moe.zip' now.")
            
            try:
                from google.colab import files
                uploaded = files.upload()
                if 'nano_moe.zip' in uploaded:
                    import zipfile
                    with zipfile.ZipFile('nano_moe.zip', 'r') as zip_ref:
                        zip_ref.extractall('.')
                    print("[SUCCESS] Extracted nano_moe.zip")
            except Exception as e:
                print(f"[ERROR] Upload widget failed: {e}")
                print("Try dragging and dropping 'nano_moe.zip' into the file explorer pane on the left.")
        else:
            print("[INFO] 'nano_moe' package found.")
        
        # Add current dir to path to find the extracted package
        if current_dir not in sys.path:
            sys.path.append(current_dir)

    else:
        # Local development
        project_root = os.path.abspath(os.path.join(current_dir, '..'))
        if project_root not in sys.path:
            sys.path.append(project_root)

setup_environment()

import torch
import torch.nn as nn
import numpy as np
from rich.console import Console

try:
    import nano_moe
    print(f"[SUCCESS] Successfully imported nano_moe from {nano_moe.__file__}")
except ImportError:
    print("[CRITICAL] Could not import nano_moe. Please check the output above.")

from nano_moe.config import TrainingConfig
from nano_moe.data.text import get_text_loaders
from nano_moe.models.phase import SparseFourierPhaseTransformer
from nano_moe.training.trainer import train_epoch, eval_model
from nano_moe.training.tracker import ExperimentTracker

console = Console()

## Configuration

In [ ]:
cfg = TrainingConfig()
cfg.dataset_type = "text"
cfg.dataset_names = ["tinyshakespeare"]
cfg.model_type = "sfpt"
cfg.batch_size = 32
cfg.seq_len = 128
cfg.epochs = 1

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Load Data

In [ ]:
loaders, vocab_size = get_text_loaders("tinyshakespeare", cfg.batch_size, cfg.seq_len)
print(f"Vocab Size: {vocab_size}")

## Initialize SFPT Model

In [ ]:
model = SparseFourierPhaseTransformer(
    vocab_size=vocab_size,
    dim=cfg.feature_dim,
    depth=cfg.depth,
    n_heads=cfg.n_heads,
    n_freqs=cfg.n_freqs,
    top_k=cfg.top_k,
    expansion=cfg.expansion
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)
criterion = nn.CrossEntropyLoss()
tracker = ExperimentTracker(cfg.save_dir)

print(model)

## Training Loop

In [ ]:
for epoch in range(1, cfg.epochs + 1):
    print(f"Starting Epoch {epoch}...")
    losses, gates, thinking = train_epoch(model, loaders, optimizer, criterion, device, ["tinyshakespeare"], tracker, epoch)
    
    accs = eval_model(model, loaders, device, ["tinyshakespeare"])
    avg_loss = np.mean(losses)
    
    print(f"Epoch {epoch} | Loss: {avg_loss:.4f} | Acc: {accs['tinyshakespeare']:.4f}")

## Text Generation Demo

In [ ]:
def generate_text(model, start_text, max_len=100):
    model.eval()
    # Need the tokenizer from get_text_loaders to encode
    # For this demo, we'll just assume we can get the tokenizer or just skip if not easily accessible in this scope
    # Re-initializing tokenizer for demo purposes
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    
    input_ids = tokenizer.encode(start_text, return_tensors="pt").to(device)
    
    generated = input_ids
    
    with torch.no_grad():
        for _ in range(max_len):
            # Forward pass
            # Note: SFPT forward expects (B, L) and returns (B, L, V) for sequence
            # We pass classification=False implicitly via our updated logic or explicitly
            logits, _, _ = model(generated, classification=False)
            
            # Get last token logits
            next_token_logits = logits[:, -1, :]
            
            # Sample
            probs = F.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            
            generated = torch.cat([generated, next_token], dim=1)
            
    return tokenizer.decode(generated[0], skip_special_tokens=True)

print("Generating text...")
try:
    sample = generate_text(model, "To be or not to be")
    print("Generated:", sample)
except Exception as e:
    print(f"Generation failed: {e}")